In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


# 1. 데이터 가져오기

In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import tensorflow.compat.v1 as tf

In [5]:

df = pd.read_csv(r'C:\ai_x\download\shareData\부동산_250213\최종전국평당분양가격(결측치제외).csv',
         encoding='cp949')

In [6]:
df.head()

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0


# 2. 지역명의 라벨인코딩
- 라벨인코딩한 지역명2 필드를 추가 / 제대로 안돌아갈 확률이 높음
- 이 후 원핫인코딩을 해보자

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [23]:
df_input = df.iloc[:,:-1]

In [8]:
df['지역명2'] = le.fit_transform(df['지역명'])
X_data = df[['지역명2', '연도', '월']].values
y_data = df[['평당분양가격']].values

In [9]:
X_data,y_data

(array([[   8, 2013,   12],
        [   7, 2013,   12],
        [   5, 2013,   12],
        ...,
        [   3, 2024,    8],
        [   2, 2024,    8],
        [  14, 2024,    8]], dtype=int64),
 array([[18189. ],
        [ 8111. ],
        [ 8080. ],
        ...,
        [13827. ],
        [13252.8],
        [25419.9]]))

# 3. 변수간 스케일 조정(min-max / std)
- 입력변수와 타겟 변수 따로 스케일 조정
- normalization : 지역명2n, 연도n, 월n 필드 추가
- standardization : 지역명2s, 연도s, 월s 필드 추가

In [10]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [35]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaled_X_data = scaler_X.fit_transform(X_data)
scaled_y_data = scaler_y.fit_transform(y_data)

In [41]:
# df_n = pd.DataFrame({'지역명2n' : [x[0] for x in scaled_X_data],
#                     '연도n' : [x[1] for x in scaled_X_data],
#                     '월n' : [x[2] for x in scaled_X_data], 
#                     '평당분양가n' : [y[0] for y in scaled_y_data]})
# df_n.head()

df[['지역명2n', '연도n', '월n']] = scaled_X_data
df[['평당분양가n']] = scaled_y_data

In [43]:
df.drop('평당분양가격n', axis=1, inplace=True)

In [44]:
# df = pd.concat([df, df_n], axis=1)
df.head()

,지역명,연도,월,평당분양가격,지역명2,지역명2n,연도n,월n,평당분양가n,지역명2s,연도s,월s,평당분양가s
0,서울,2013,12,18189.0,8,0.5000,0.0,1.0,0.328198,0.000000,-1.875367,1.62196,1.168591
1,부산,2013,12,8111.0,7,0.4375,0.0,1.0,0.065274,-0.204124,-1.875367,1.62196,-0.728312
2,대구,2013,12,8080.0,5,0.3125,0.0,1.0,0.064466,-0.612372,-1.875367,1.62196,-0.734147
3,인천,2013,12,10204.0,11,0.6875,0.0,1.0,0.119878,0.612372,-1.875367,1.62196,-0.334363
4,광주,2013,12,6098.0,4,0.2500,0.0,1.0,0.012757,-0.816497,-1.875367,1.62196,-1.107203


In [15]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

Instructions for updating:
non-resource variables are not supported in the long term


In [16]:
# placeholder 설정 (입력변수 x, 타겟변수 y를 나중에)
X = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

# W, b
W = tf.Variable(tf.random.normal([1]), name = 'weight')
b = tf.Variable(tf.random.normal([1]), name = 'bias')

# Hypothesis(예측값)
H = W*X + b

# 손실함수
cost = tf.reduce_mean(tf.square(H-y))

# 경사하강법
train = tf.train.GradientDescentOptimizer(learning_rate=0.001).minimize(cost)

# 세션 객체 생성
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # W, b 초기화

# n번 학습
for step in range(30001):
    _, cost_val, W_val, b_val = sess.run([train, cost, W, b], feed_dict={X:scaled_X_data, 
                                                                         y:scaled_y_data})
    if step%3000 ==0:
        print("{}째 : cost : {}, W:{}, b : {}".format(step,
                                                    cost_val,
                                                    W_val,
                                                    b_val))

0째 : cost : 1.6175527572631836, W:[-1.3785018], b : [-0.31867632]
3000째 : cost : 0.05237746238708496, W:[-0.53593016], b : [0.46072856]
6000째 : cost : 0.03307922184467316, W:[-0.32609203], b : [0.3467847]
9000째 : cost : 0.02487506903707981, W:[-0.18943168], b : [0.27220055]
12000째 : cost : 0.021387211978435516, W:[-0.10032612], b : [0.22356993]
15000째 : cost : 0.01990441232919693, W:[-0.04222738], b : [0.19186169]
18000째 : cost : 0.0192740298807621, W:[-0.0043458], b : [0.17118727]
21000째 : cost : 0.01900602877140045, W:[0.02035382], b : [0.15770708]
24000째 : cost : 0.018892094492912292, W:[0.03645848], b : [0.1489177]
27000째 : cost : 0.01884365640580654, W:[0.04695905], b : [0.14318694]
30000째 : cost : 0.018823063001036644, W:[0.05380559], b : [0.13945042]


In [37]:
# w 변수를 분화해서 계산하기

# placeholder 설정 (입력변수 x, 타겟변수 y를 나중에)
X = tf.placeholder(dtype=tf.float32, shape=[None, 3])
y = tf.placeholder(dtype=tf.float32, shape=[None, 1])

# W, b
W = tf.Variable(tf.random.normal([3, 1]), name='weight')
b = tf.Variable(tf.random.normal([1]), name='bias')

# Hypothesis(예측값)
# H = W1*X[0]+W2*X[1]+W2*X[2] + b
H = tf.matmul(X, W) + b

# 손실함수
cost = tf.reduce_mean(tf.square(H-y))

# 경사하강법
train = tf.train.GradientDescentOptimizer(learning_rate=0.001).minimize(cost)

# 세션 객체 생성
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # W, b 초기화

# W 따로 보기



# n번 학습
for step in range(50001):
    _, cost_val, W_val, b_val = sess.run([train, cost, W, b], feed_dict={X:scaled_X_data, 
                                                                         y:scaled_y_data})
    W1_val, W2_val, W3_val = W_val.reshape(-1)
    if step%10000 ==0:
        print("{}째 : cost : {}, W1:{}, W2:{}, W3:{}, b : {}".format(step,
                                                    cost_val,
                                                    W1_val, W2_val, W3_val,
                                                    b_val))

0째 : cost : 7.451117038726807, W1:-1.8386859893798828, W2:-1.8596160411834717, W3:-0.3275931179523468, b : [-0.3600072]
10000째 : cost : 0.03567872568964958, W1:-0.31775522232055664, W2:-0.1482812911272049, W3:-0.08233880251646042, b : [0.4638198]
20000째 : cost : 0.016455814242362976, W1:-0.12815701961517334, W2:0.11836311221122742, W3:-0.05801914259791374, b : [0.20401996]
30000째 : cost : 0.013792327605187893, W1:-0.06843364238739014, W2:0.21193042397499084, W3:-0.017511898651719093, b : [0.09991915]
40000째 : cost : 0.013379986397922039, W1:-0.04696628823876381, W2:0.24746108055114746, W3:0.0027706085238605738, b : [0.05849471]
50000째 : cost : 0.013315228745341301, W1:-0.03877267241477966, W2:0.2613508403301239, W3:0.011391837149858475, b : [0.04204749]


In [17]:
W_, b_ = sess.run([W,b])
W_[0], b_[0]

[array([0.05380559], dtype=float32), array([0.13945042], dtype=float32)]

In [18]:
# 표준화
scaler_X = StandardScaler()
scaler_y = StandardScaler()
scaled_X_data = scaler_X.fit_transform(X_data)
scaled_y_data = scaler_y.fit_transform(y_data)

In [27]:
scaled_X_data, scaled_y_data

(array([[ 0.        , -1.87536661,  1.62196025],
        [-0.20412415, -1.87536661,  1.62196025],
        [-0.61237244, -1.87536661,  1.62196025],
        ...,
        [-1.02062073,  1.66419932,  0.46374038],
        [-1.22474487,  1.66419932,  0.46374038],
        [ 1.22474487,  1.66419932,  0.46374038]]),
 array([[ 1.16859132],
        [-0.72831216],
        [-0.73414705],
        ...,
        [ 0.34756602],
        [ 0.23948883],
        [ 2.52960734]]))

In [20]:
df_s = pd.DataFrame({'지역명2s' : [x[0] for x in scaled_X_data],
                    '연도s' : [x[1] for x in scaled_X_data],
                    '월s' : [x[2] for x in scaled_X_data], 
                    '평당분양가s' : [y[0] for y in scaled_y_data]})
df_s.head()

,지역명2s,연도s,월s,평당분양가s
0,0.000000,-1.875367,1.62196,1.168591
1,-0.204124,-1.875367,1.62196,-0.728312
2,-0.612372,-1.875367,1.62196,-0.734147
3,0.612372,-1.875367,1.62196,-0.334363
4,-0.816497,-1.875367,1.62196,-1.107203


In [23]:
df = pd.concat([df, df_s], axis = 1)


In [34]:
# placeholder 설정 (입력변수 x, 타겟변수 y를 나중에)
X = tf.placeholder(dtype=tf.float32, shape=[None, 3])
y = tf.placeholder(dtype=tf.float32, shape=[None, 1])

# W, b
W = tf.Variable(tf.random.normal([3, 1]), name='weight')
b = tf.Variable(tf.random.normal([1]), name='bias')

# Hypothesis(예측값)
# H = W*x + b
H = tf.matmul(X, W) + b

# 손실함수
cost = tf.reduce_mean(tf.square(H-y))

# 경사하강법
train = tf.train.GradientDescentOptimizer(learning_rate=0.001).minimize(cost)

# 세션 객체 생성
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # W, b 초기화

# W 따로 보기



# n번 학습
for step in range(30001):
    _, cost_val, W_val, b_val = sess.run([train, cost, W, b], feed_dict={X:scaled_X_data, 
                                                                         y:scaled_y_data})
    W1_val, W2_val, W3_val = W_val.reshape(-1)
    if step%3000 ==0:
        print("{}째 : cost : {}, W1:{}, W2:{}, W3:{}, b : {}".format(step,
                                                    cost_val,
                                                    W1_val, W2_val, W3_val,
                                                    b_val))

0째 : cost : 3.113179922103882, W1:-0.3926301598548889, W2:-0.301180362701416, W3:1.2134732007980347, b : [-0.2223299]
3000째 : cost : 0.6924510598182678, W1:-0.07473745942115784, W2:0.5502879619598389, W3:0.041189491748809814, b : [-0.0005478]
6000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.038989510387182236, b : [-1.3568254e-06]
9000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.03898535668849945, b : [-2.3496249e-08]
12000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.03898535668849945, b : [-1.8333372e-08]
15000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.03898535668849945, b : [-1.6937156e-08]
18000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.03898535668849945, b : [-1.554094e-08]
21000째 : cost : 0.6924440860748291, W1:-0.07395415008068085, W2:0.5512243509292603, W3:0.03898535668849945, b : [-

array([[11980.427, 10415.997, 13333.466],
       [11810.146, 10415.997, 13333.466],
       [11469.586, 10415.997, 13333.466],
       ...,
       [11129.025, 13368.701, 12367.279],
       [10958.745, 13368.701, 12367.279],
       [13002.108, 13368.701, 12367.279]], dtype=float32)


## 원핫인코딩


In [38]:
from tensorflow.keras.utils import to_categorical 

In [45]:
categorical_one_hot = to_categorical(df['지역명2'])
categorical_one_hot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)

In [52]:
# 지역명, 지역명 2
loc_info = df[['지역명', '지역명2']].head(17).sort_values(by='지역명2')
loc_column_names = loc_info['지역명'].tolist()

In [53]:
df[loc_column_names] = categorical_one_hot
df

,지역명,연도,월,평당분양가격,지역명2,지역명2n,연도n,월n,평당분양가n,지역명2s,...,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,서울,2013,12,18189.0,8,0.5000,0.0,1.000000,0.328198,0.000000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,부산,2013,12,8111.0,7,0.4375,0.0,1.000000,0.065274,-0.204124,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,대구,2013,12,8080.0,5,0.3125,0.0,1.000000,0.064466,-0.612372,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,인천,2013,12,10204.0,11,0.6875,0.0,1.000000,0.119878,0.612372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,광주,2013,12,6098.0,4,0.2500,0.0,1.000000,0.012757,-0.816497,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,전북,2024,8,12058.2,13,0.8125,1.0,0.636364,0.168252,1.020621,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2172,전남,2024,8,13120.8,12,0.7500,1.0,0.636364,0.195974,0.816497,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2173,경북,2024,8,13827.0,3,0.1875,1.0,0.636364,0.214398,-1.020621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2174,경남,2024,8,13252.8,2,0.1250,1.0,0.636364,0.199418,-1.224745,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
pd.get_dummies(df['지역명'])

,강원,경기,경남,경북,광주,대구,대전,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2172,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2173,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2174,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
